# AlphaFold Ensemble Competition - Tournament Edition

This notebook screens a list of potential peptide binders using a 'Winner Stays On' (King of the Hill) tournament structure.
It evaluates pairs of binders against the target protein. The winner advances to face the next candidate on the list.

In [1]:
import os
import time
import py3Dmol
import numpy as np
from pathlib import Path
from af_competition import run_colabfold_async, analyze_binding, process_ensemble, optimize_threshold  # noqa: F401


## 1. Configuration
Define the target, binding site, and list of potential binders.

In [2]:
TARGET_SEQ = "SQIPASEQETLVRPKPLLLKLLKSVGAQKDTYTMKEVLFYLGQYIMTKRLYDAAQQHIVYCSNDLLGDLFGVPSFSVKEHRKIYTMIYRNLVVVNQQ"
BINDING_SITE_RESIDUES = [42, 84]  # 1-indexed on Target Chain

# List of potential binders to screen in the tournament
BINDER_CANDIDATES = [
    "ETFSDLWKLLPE", # Candidate 0
    "LTFEHYWAQLTS", # Candidate 1
    "LTWEHYWAQLTS", 
    "LTFEHYLAQLTS", 
    "LTFEHIWAQLTS", 
    "LTFEHAFAQLTS", 
    "LTFEDYTAQFTS"
]

NUM_SEEDS = 20
BASE_OUTPUT_DIRECTORY = "./colabfold_results"
MIN_PLDDT = 80.0


## 2. Tournament Execution
Runs the 'Winner Stays On' tournament.

In [3]:
champion_idx = 0
champion_seq = BINDER_CANDIDATES[0]

last_match_dir = ""
last_match_stats = {}
winning_state_last_match = ""
majority_wins = (NUM_SEEDS // 2) + 1

print(f"Starting Tournament with {len(BINDER_CANDIDATES)} candidates. Early stop threshold: {majority_wins} wins.\n")

for challenger_idx in range(1, len(BINDER_CANDIDATES)):
    challenger_seq = BINDER_CANDIDATES[challenger_idx]
    run_name = f"tournament_match_{champion_idx}_vs_{challenger_idx}"
    
    print(f"--- Match {challenger_idx}: Champion [{champion_idx}] vs Challenger [{challenger_idx}] ---")
    
    # --- 1. Execute ColabFold Asynchronously ---
    process, ACTUAL_OUTPUT_DIR, log_file = run_colabfold_async(TARGET_SEQ, champion_seq, challenger_seq, BASE_OUTPUT_DIRECTORY, run_name=run_name, num_seeds=NUM_SEEDS)
    last_match_dir = ACTUAL_OUTPUT_DIR
    
    # --- 2. Monitor and Analyze in Real-Time ---
    analyzed_pdbs = set()
    all_results = []
    early_stop_triggered = False
    
    while process.poll() is None:
        current_pdbs = set(Path(ACTUAL_OUTPUT_DIR).glob("*.pdb"))
        new_pdbs = current_pdbs - analyzed_pdbs
        
        newly_analyzed = 0
        for pdb in new_pdbs:
            try:
                res = analyze_binding(str(pdb), BINDING_SITE_RESIDUES, 0.0)
                if res["mean_plddt"] >= MIN_PLDDT:
                    all_results.append(res)
                analyzed_pdbs.add(pdb)
                newly_analyzed += 1
            except Exception:
                # File might be partially written; skip and retry next loop
                pass
                
        if newly_analyzed > 0 and all_results:
            opt = optimize_threshold(all_results, min_thresh=5.0, max_thresh=15.0, step=0.5)
            stats = opt['stats']
            champ_wins = stats.get('lig1_wins', 0)
            challenger_wins = stats.get('lig2_wins', 0)
            
            print(f"Models parsed: {len(analyzed_pdbs)}/{NUM_SEEDS} | Champion: {champ_wins} | Challenger: {challenger_wins}")
            
            if champ_wins >= majority_wins or challenger_wins >= majority_wins:
                print("  -> Insurmountable lead detected! Terminating AlphaFold early...")
                process.terminate()
                if 'log_file' in locals():
                    log_file.close()
                early_stop_triggered = True
                break
                
        time.sleep(5)  # Wait 5 seconds before checking again
        
    # Parse any final straggler PDBs after process ends
    current_pdbs = set(Path(ACTUAL_OUTPUT_DIR).glob("*.pdb"))
    new_pdbs = current_pdbs - analyzed_pdbs
    for pdb in new_pdbs:
        try:
            res = analyze_binding(str(pdb), BINDING_SITE_RESIDUES, 0.0)
            if res["mean_plddt"] >= MIN_PLDDT:
                all_results.append(res)
        except Exception:
            pass
            
    if 'log_file' in locals() and not log_file.closed:
        log_file.close()
    
    if not all_results:
        print(f"Match void: No models passed pLDDT threshold {MIN_PLDDT}.")
        print(f"Champion [{champion_idx}] retains title by default.\n")
        winning_state_last_match = "Ligand 1"
        continue
        
    # Final optimization across all parsed results
    opt = optimize_threshold(all_results, min_thresh=5.0, max_thresh=15.0, step=0.5)
    stats = opt['stats']
    last_match_stats = stats
    
    champ_wins = stats.get('lig1_wins', 0)
    challenger_wins = stats.get('lig2_wins', 0)
    
    print(f"\nFinal Match Results (Valid Models: {len(all_results)})")
    print(f"Champion score: {champ_wins} | Challenger score: {challenger_wins}")
    
    if challenger_wins > champ_wins:
        print(f"Challenger [{challenger_idx}] defeats Champion [{champion_idx}]!")
        champion_idx = challenger_idx
        champion_seq = challenger_seq
        winning_state_last_match = "Ligand 2"
    elif champ_wins > challenger_wins:
        print(f"Champion [{champion_idx}] defends the title!")
        winning_state_last_match = "Ligand 1"
    else:
        # TIE BREAKER
        print("Scores tied! Proceeding to pLDDT tie-breaker...")
        if champ_wins == 0 and challenger_wins == 0:
            print("Neither ligand achieved exclusive binding in any valid models. Champion retains title by default.")
            winning_state_last_match = "Neither"
        else:
            champ_plddt = np.mean([r['mean_plddt'] for r in stats['lig1_results']])
            challenger_plddt = np.mean([r['mean_plddt'] for r in stats['lig2_results']])
            print(f"Champion Avg pLDDT: {champ_plddt:.1f} | Challenger Avg pLDDT: {challenger_plddt:.1f}")
            
            if challenger_plddt > champ_plddt:
                print(f"Challenger [{challenger_idx}] wins by tie-breaker!")
                champion_idx = challenger_idx
                champion_seq = challenger_seq
                winning_state_last_match = "Ligand 2"
            else:
                print(f"Champion [{champion_idx}] wins by tie-breaker!")
                winning_state_last_match = "Ligand 1"
    print("\n")
    
print("=== TOURNAMENT COMPLETE ===")
print(f"ULTIMATE CHAMPION: Candidate [{champion_idx}] ({champion_seq})")


Starting Tournament with 7 candidates. Early stop threshold: 11 wins.

--- Match 1: Champion [0] vs Challenger [1] ---


E0000 00:00:1788536186.391910 2773289 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788536186.398588 2773289 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788536186.416471 2773289 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788536186.416488 2773289 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788536186.416506 2773289 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788536186.416509 2773289 computation_placer.cc:177] computation placer already registered. Please check linka

2026-09-04 16:36:29,394 Running colabfold 1.6.1 (de5ab5f795ed95c70a7a9b6a9dc6bb5625016142)
2026-09-04 16:36:29,762 Running on GPU
2026-09-04 16:36:29,942 Found 5 citations for tools or databases
2026-09-04 16:36:29,942 Query 1/1: complex (length 121)


COMPLETE: 100%|██████████| 450/450 [elapsed: 00:01 remaining: 00:00]


Models parsed: 1/20 | Champion: 0 | Challenger: 1
Models parsed: 3/20 | Champion: 0 | Challenger: 3
Models parsed: 4/20 | Champion: 0 | Challenger: 4
Models parsed: 6/20 | Champion: 0 | Challenger: 6
Models parsed: 8/20 | Champion: 0 | Challenger: 8
Models parsed: 9/20 | Champion: 0 | Challenger: 9
Models parsed: 11/20 | Champion: 0 | Challenger: 11
  -> Insurmountable lead detected! Terminating AlphaFold early...

Final Match Results (Valid Models: 11)
Champion score: 0 | Challenger score: 11
Challenger [1] defeats Champion [0]!


--- Match 2: Champion [1] vs Challenger [2] ---


...


2026-09-04 16:54:24,919 alphafold2_multimer_v3_model_1_seed_141084 recycle=1 pLDDT=85.1 pTM=0.803 ipTM=0.738 tol=0.533
2026-09-04 16:54:25,662 alphafold2_multimer_v3_model_1_seed_141084 recycle=2 pLDDT=85.9 pTM=0.821 ipTM=0.77 tol=0.263
2026-09-04 16:54:25,663 alphafold2_multimer_v3_model_1_seed_141084 took 2.2s (2 recycles)
2026-09-04 16:54:26,414 alphafold2_multimer_v3_model_1_seed_141085 recycle=0 pLDDT=75.9 pTM=0.722 ipTM=0.314
2026-09-04 16:54:27,150 alphafold2_multimer_v3_model_1_seed_141085 recycle=1 pLDDT=78.5 pTM=0.744 ipTM=0.35 tol=2.5
2026-09-04 16:54:27,883 alphafold2_multimer_v3_model_1_seed_141085 recycle=2 pLDDT=79.4 pTM=0.748 ipTM=0.385 tol=3.03
2026-09-04 16:54:28,616 alphafold2_multimer_v3_model_1_seed_141085 recycle=3 pLDDT=83.8 pTM=0.792 ipTM=0.631 tol=0.66
2026-09-04 16:54:29,352 alphafold2_multimer_v3_model_1_seed_141085 recycle=4 pLDDT=85 pTM=0.812 ipTM=0.726 tol=0.472
2026-09-04 16:54:29,352 alphafold2_multimer_v3_model_1_seed_141085 took 3.7s (4 recycles)
2026

E0000 00:00:1788537522.736348 2800720 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788537522.743614 2800720 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788537522.762251 2800720 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788537522.762269 2800720 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788537522.762271 2800720 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788537522.762288 2800720 computation_placer.cc:177] computation placer already registered. Please check linka

2026-09-04 16:58:45,984 Running colabfold 1.6.1 (de5ab5f795ed95c70a7a9b6a9dc6bb5625016142)
2026-09-04 16:58:46,354 Running on GPU
2026-09-04 16:58:46,540 Found 5 citations for tools or databases
2026-09-04 16:58:46,540 Query 1/1: complex (length 121)


COMPLETE: 100%|██████████| 450/450 [elapsed: 00:01 remaining: 00:00]


2026-09-04 16:54:30,095 Setting max_seq=508, max_extra_seq=1015
2026-09-04 16:56:43,510 alphafold2_multimer_v3_model_1_seed_176698 recycle=0 pLDDT=84.4 pTM=0.798 ipTM=0.67
2026-09-04 16:58:13,471 alphafold2_multimer_v3_model_1_seed_176698 recycle=1 pLDDT=85.6 pTM=0.819 ipTM=0.76 tol=0.502
2026-09-04 16:58:14,208 alphafold2_multimer_v3_model_1_seed_176698 recycle=2 pLDDT=86.2 pTM=0.819 ipTM=0.758 tol=0.7
2026-09-04 16:58:14,945 alphafold2_multimer_v3_model_1_seed_176698 recycle=3 pLDDT=86.2 pTM=0.829 ipTM=0.769 tol=0.69
2026-09-04 16:58:15,682 alphafold2_multimer_v3_model_1_seed_176698 recycle=4 pLDDT=86.6 pTM=0.821 ipTM=0.771 tol=0.542
2026-09-04 16:58:16,419 alphafold2_multimer_v3_model_1_seed_176698 recycle=5 pLDDT=86.4 pTM=0.826 ipTM=0.776 tol=0.49
2026-09-04 16:58:16,420 alphafold2_multimer_v3_model_1_seed_176698 took 219.6s (5 recycles)
2026-09-04 16:58:17,179 alphafold2_multimer_v3_model_1_seed_176699 recycle=0 pLDDT=83.6 pTM=0.79 ipTM=0.69
2026-09-04 16:58:17,916 alphafold2_mult

## 3. Visualization of the Final Match
Renders the highest confidence model of the Ultimate Champion winning its last match.

In [4]:
if not last_match_stats:
    print("No valid models were generated to visualize.")
elif winning_state_last_match == "Neither":
    print("Neither ligand bound in the final match, nothing to visualize.")
else:
    winning_results = []
    
    if winning_state_last_match == "Ligand 1":
        winning_results = last_match_stats.get('lig1_results', [])
    elif winning_state_last_match == "Ligand 2":
        winning_results = last_match_stats.get('lig2_results', [])
        
    if not winning_results:
        print("No structural models available for the winning state to render.")
    else:
        best_model = max(winning_results, key=lambda x: x["mean_plddt"])
        best_pdb = os.path.join(last_match_dir, best_model["pdb_file"])
        
        print(f"Visualizing Final Match. Winning State: {winning_state_last_match}")
        print(f"Best Model: {best_model['pdb_file']} (pLDDT: {best_model['mean_plddt']:.1f})")
        print("Target (Chain A) = Grey | Ligand 1 (Champion) = Blue | Ligand 2 (Challenger) = Red")
        
        if os.path.exists(best_pdb):
            with open(best_pdb, 'r') as f:
                pdb_data = f.read()
                
            view = py3Dmol.view(width=800, height=600)
            view.addModel(pdb_data, 'pdb')
            
            # Target (Chain A)
            view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray'}})
            # Ligand 1 (Chain B)
            view.setStyle({'chain': 'B'}, {'cartoon': {'color': 'blue'}, 'stick': {'color': 'blue'}})
            # Ligand 2 (Chain C)
            view.setStyle({'chain': 'C'}, {'cartoon': {'color': 'red'}, 'stick': {'color': 'red'}})
            
            view.zoomTo()
            view.show()
        else:
            print(f"Error: Cannot find {best_pdb} to visualize.")


Visualizing Final Match. Winning State: Ligand 1
Best Model: complex_unrelaxed_alphafold2_multimer_v3_model_1_seed_489430.pdb (pLDDT: 86.2)
Target (Chain A) = Grey | Ligand 1 (Champion) = Blue | Ligand 2 (Challenger) = Red


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

2026-09-04 16:58:50,997 Setting max_seq=508, max_extra_seq=1015
2026-09-04 17:01:08,246 alphafold2_multimer_v3_model_1_seed_489424 recycle=0 pLDDT=82.1 pTM=0.767 ipTM=0.647
2026-09-04 17:02:39,083 alphafold2_multimer_v3_model_1_seed_489424 recycle=1 pLDDT=84.2 pTM=0.797 ipTM=0.73 tol=1.14
2026-09-04 17:02:39,822 alphafold2_multimer_v3_model_1_seed_489424 recycle=2 pLDDT=85 pTM=0.808 ipTM=0.745 tol=0.452
2026-09-04 17:02:39,823 alphafold2_multimer_v3_model_1_seed_489424 took 221.6s (2 recycles)
2026-09-04 17:02:40,575 alphafold2_multimer_v3_model_1_seed_489425 recycle=0 pLDDT=82.3 pTM=0.781 ipTM=0.679
2026-09-04 17:02:41,311 alphafold2_multimer_v3_model_1_seed_489425 recycle=1 pLDDT=83.9 pTM=0.803 ipTM=0.744 tol=0.989
2026-09-04 17:02:42,046 alphafold2_multimer_v3_model_1_seed_489425 recycle=2 pLDDT=86.1 pTM=0.818 ipTM=0.757 tol=0.834
2026-09-04 17:02:42,787 alphafold2_multimer_v3_model_1_seed_489425 recycle=3 pLDDT=85.9 pTM=0.812 ipTM=0.762 tol=0.281
2026-09-04 17:02:42,788 alphafold2_